# 라즈베리파이에서 YOLO11 사용하기

### YOLO11은 무엇인가

YOLO11은 이미지나 영상 속 물체의 위치와 종류를 찾아주는 객체 탐지 모델이다. 화면 속에 사람이 있으면 사람 주변에 사각형 박스를 그리고 `person`이라는 이름과 신뢰도 점수를 함께 표시한다. 라즈베리파이에서는 큰 모델보다 가장 가벼운 `yolo11n.pt`로 시작하는 것이 적절하다.


### 라즈베리파이에서 작은 모델을 사용하는 이유

라즈베리파이는 노트북이나 데스크톱보다 연산 성능이 낮다. 그래서 정확도가 높은 큰 모델을 사용하면 화면이 매우 느려질 수 있다. 이번 실습에서는 속도와 설치 난이도를 고려해 `YOLO11n`을 기본 모델로 사용한다.


### 실습 환경 확인

먼저 현재 Python 버전과 실행 환경을 확인한다. 라즈베리파이에서 실행 중인지, Python이 정상적으로 동작하는지 확인하는 단계이다.

In [ ]:
import sys, platform, os

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

### 패키지 설치

YOLO11 실행에는 `ultralytics`, 이미지 표시에는 `pillow`, 영상 배열 처리를 위해 `opencv-python`을 사용한다. 이미 설치되어 있다면 이 셀을 다시 실행해도 된다. 단, 교육장 환경에서 패키지가 미리 설치되어 있으면 이 셀은 건너뛰어도 된다.


In [ ]:
# 필요한 패키지 설치
# 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.
!python -m pip install -U pip
!python -m pip install ultralytics pillow opencv-python

### 라이브러리 불러오기

실습에 필요한 Python 라이브러리를 불러온다. 업로드한 노트북과 동일하게 `pinkylib`의 `Camera` 클래스를 사용하고, 카메라 프레임을 화면에 표시하기 위해 `PIL.Image`와 `IPython.display`를 사용한다.

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display, clear_output
import time
import io
import os

# 업로드한 my_petbot.ipynb에서 사용한 카메라 방식
from pinkylib import Camera

### 카메라 사용 방식

카메라를 `cam = Camera()`로 생성하고, `cam.start()`로 시작한 뒤, `cam.get_frame()`으로 한 장의 프레임을 가져온다. 사용이 끝나면 `cam.close()`로 카메라를 닫는다. 가져온 프레임은 BGR 순서이므로, 사람이 보기 좋게 표시할 때는 RGB 순서로 바꿔준다.

In [ ]:
def capture_one_frame(cam):
    """pinkylib.Camera에서 프레임 한 장을 가져옵니다.

    업로드한 노트북 기준:
    - cam.get_frame()으로 프레임 획득
    - 프레임은 BGR 순서
    - 화면 표시용으로는 RGB로 변환
    """
    frame_bgr = cam.get_frame()
    image_rgb = Image.fromarray(frame_bgr[:, :, ::-1])
    return frame_bgr, image_rgb


def capture_one_frame_bytes(cam):
    """카메라 프레임을 JPEG bytes로 변환합니다.

    OpenAI Vision API나 파일 저장 등에 사용할 수 있는 형태입니다.
    """
    frame_bgr, image_rgb = capture_one_frame(cam)
    buf = io.BytesIO()
    image_rgb.save(buf, format="JPEG")
    return buf.getvalue()

### 카메라 한 장 테스트

YOLO를 실행하기 전에 카메라가 정상적으로 작동하는지 확인한다. 이 테스트가 실패하면 YOLO 문제가 아니라 카메라 연결, 카메라 권한, 카메라 라이브러리 문제일 가능성이 높다.

In [ ]:
def test_camera_once():
    cam = Camera()
    cam.start()
    try:
        frame_bgr, image_rgb = capture_one_frame(cam)
        display(image_rgb)
        return frame_bgr
    finally:
        cam.close()

last_frame = test_camera_once()

### YOLO11n 모델 불러오기

카메라가 정상적으로 동작하면 YOLO11n 모델을 불러온다. 처음 실행할 때는 모델 파일이 자동으로 다운로드될 수 있다. 인터넷이 없는 환경에서는 미리 `yolo11n.pt` 파일을 준비해 노트북과 같은 폴더에 넣어야 한다.

In [ ]:
model = YOLO("yolo11n.pt")
print("Model loaded:", model.model_name if hasattr(model, "model_name") else "yolo11n.pt")

### 카메라 사진 한 장에 YOLO11 적용하기

카메라에서 한 장의 이미지를 가져와 YOLO11에 입력한다. 탐지 결과는 박스가 그려진 이미지로 출력된다. 이 셀은 전체 실습의 핵심이며, 라즈베리파이 카메라와 YOLO11이 연결되는 부분이다.

In [ ]:
def detect_once_from_camera(imgsz=640, conf=0.25):
    cam = Camera()
    cam.start()
    try:
        frame_bgr, image_rgb = capture_one_frame(cam)
        display(image_rgb)

        results = model.predict(
            source=frame_bgr,
            imgsz=imgsz,
            conf=conf,
            verbose=False
        )

        annotated_bgr = results[0].plot()
        annotated_rgb = Image.fromarray(annotated_bgr[:, :, ::-1])
        display(annotated_rgb)

        return results[0], annotated_bgr
    finally:
        cam.close()

last_result, last_annotated_bgr = detect_once_from_camera(imgsz=640, conf=0.25)

### 탐지 결과 읽기

YOLO 결과에는 물체 이름, 신뢰도, 박스 좌표가 들어 있다. 신뢰도는 모델이 그 물체라고 얼마나 확신하는지를 나타낸다. 예를 들어 `person 0.82`는 사람일 가능성이 82% 정도라는 의미로 이해하면 된다.


In [ ]:
def print_detection_result(result):
    if result.boxes is None or len(result.boxes) == 0:
        print("탐지된 물체가 없습니다.")
        return

    for i, box in enumerate(result.boxes, start=1):
        cls_id = int(box.cls[0])
        name = model.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
        print(f"{i}. {name} / confidence={conf:.2f} / box=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

print_detection_result(last_result)

### 짧은 실시간 탐지 실행하기

카메라 프레임을 여러 번 가져와 YOLO11을 반복 실행하면 짧은 실시간 객체 탐지처럼 사용할 수 있다. 주피터 노트북에서는 화면 출력이 느릴 수 있으므로 무한 반복 대신 정해진 횟수만 실행한다. 멈추고 싶으면 노트북의 정지 버튼을 누른다.


In [ ]:
def run_short_realtime_detection(num_frames=30, imgsz=320, conf=0.25, delay=0.03):
    cam = Camera()
    cam.start()
    try:
        for i in range(num_frames):
            frame_bgr, _ = capture_one_frame(cam)
            results = model.predict(
                source=frame_bgr,
                imgsz=imgsz,
                conf=conf,
                verbose=False
            )

            annotated_bgr = results[0].plot()
            annotated_rgb = Image.fromarray(annotated_bgr[:, :, ::-1])

            clear_output(wait=True)
            print(f"frame {i + 1}/{num_frames} | imgsz={imgsz} | conf={conf}")
            display(annotated_rgb)

            time.sleep(delay)
    finally:
        cam.close()

run_short_realtime_detection(num_frames=30, imgsz=320, conf=0.25)

### 입력 크기로 속도 조절하기

입력 크기인 `imgsz`를 줄이면 속도가 빨라질 수 있다. 대신 작은 물체를 놓칠 가능성이 커진다. 라즈베리파이에서는 `imgsz=320`으로 먼저 테스트하고, 여유가 있으면 `480` 또는 `640`으로 올리는방식이 좋다.

In [ ]:
def compare_imgsz_once():
    cam = Camera()
    cam.start()
    try:
        frame_bgr, image_rgb = capture_one_frame(cam)
        display(image_rgb)
    finally:
        cam.close()

    for size in [320, 480, 640]:
        t0 = time.perf_counter()
        result = model.predict(source=frame_bgr, imgsz=size, conf=0.25, verbose=False)[0]
        elapsed = time.perf_counter() - t0
        fps = 1 / elapsed if elapsed > 0 else 0
        print(f"imgsz={size} | time={elapsed:.3f} sec | approx FPS={fps:.2f} | detections={len(result.boxes)}")

compare_imgsz_once()

### 탐지 결과 이미지 저장하기

저장된 이미지는 `yolo11_result.jpg`라는 이름으로 현재 폴더에 만들어진다.

In [ ]:
# 마지막 탐지 결과 이미지 저장
# OpenCV 없이도 PIL로 저장할 수 있도록 RGB로 변환해서 저장합니다.
if "last_annotated_bgr" in globals():
    save_img = Image.fromarray(last_annotated_bgr[:, :, ::-1])
    save_path = "yolo11_result.jpg"
    save_img.save(save_path)
    print("Saved:", os.path.abspath(save_path))
    display(save_img)
else:
    print("먼저 '카메라 사진 한 장에 YOLO11 적용하기' 셀을 실행하세요.")